Refactored brain atlas alignment script.

[Runtime: <1 min per file]

Notes:
- The coordinates chosen for the QC overlay outputs (i.e. the X/Y/Z coordinates for the slices shown) will vary across masks, because these coordinates are calculated based on the non-zero voxel distributions of the atlas masks (not the subjects' brain masks) -- so this is expected behavior, not indicative of a processing problem as far as I'm aware.

---------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, csv, json, math, glob, re, time
from pathlib import Path
import subprocess
import shutil
import tempfile
from tempfile import gettempdir as _gettempdir
from typing import Optional
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt

from nilearn.image import resample_to_img, mean_img
from nilearn.datasets import load_mni152_template
from nilearn.maskers import NiftiMasker
import nilearn.plotting as plotting
from nilearn import datasets

import ants

In [ ]:
### LOAD FREESURFER:
FREESURFER_HOME = config['freesurfer']['home']
FREESURFER_LICENSE = Path(config['freesurfer'].get('license'))
FREESURFER_SUBJECTS_DIR = config['freesurfer']['subjects_dir']
os.environ['FREESURFER_HOME'] = str(FREESURFER_HOME)
os.environ['SUBJECTS_DIR']    = str(FREESURFER_SUBJECTS_DIR)
os.environ['PATH'] = f"{str(FREESURFER_HOME)}/bin:" + os.environ.get('PATH', '')
if FREESURFER_LICENSE.is_file():
    os.environ['FS_LICENSE'] = str(FREESURFER_LICENSE)
else:
    raise RuntimeError("FreeSurfer license not found: check that 'license.txt' exists in home FreeSurfer directory."
    "in FreeSurfer $HOME directory and that a valid path is set in 'config.yaml' file.")
subprocess.run("recon-all --version", shell=True, check=True)

In [ ]:
# ============================
# GLOBAL PARAMETERS
# ============================

# General:
SUBSET      = config['subset']
HARD_STOP   = config['hard_errors']
RANDOM_SEED = config['random_seed']

# Optional filtering params:
FILTER_SUBJECT_IDS = config["filter"].get("subject_IDs", [])
FILTER_SESSION_IDS = config["filter"].get("session_IDs", [])
FILTER_GROUP_IDS   = config["filter"].get("group_IDs", [])

# Alignment toggles:
OVERWRITE_TRANSFORMS  = config['overwrite_alignment_transforms']
OVERWRITE_ATLAS_CACHE = config['overwrite_atlas_cache']
SAVE_T1_ATLAS         = config['save_T1_atlas']
SAVE_INTERMEDIATE_QC  = config['save_intermediate_QC']

REDIRECT_TEMP_TOGGLE  = config['redirect_temp_toggle']
REDIRECT_TEMP_FOLDER  = config['redirect_temp_folder']
WIPE_TEMP_AFTER_RUN   = config.get('wipe_temp_after_run', False)

# Registration type:
REG_TYPE = config['registration_parameters']['registration_type']  # e.g. "SyN"

# Atlas / parcellation configuration:
PARCELLATION  = config['parcellation']
ATLAS_FAMILY  = str(PARCELLATION['atlas']).strip()      # "Craddock" | "Schaefer" | "MIST"
ATLAS_N_ROIS  = int(PARCELLATION['n_rois'])
ATLAS_NETWORK_SCALE = PARCELLATION.get('network_scale', None)
CANONICAL_RES_MM = int(config.get("atlases", {}).get("Schaefer", {}).get("resolution", 2)) # used for harmonization


atl_cfg = config['atlases']
CANONICAL_MNI = atl_cfg['Craddock']['canonical_mni']   # label for bookkeeping

AF_LOWER = ATLAS_FAMILY.lower()

## Atlas / parcellation configuration:
PARCELLATION        = config['parcellation']
ATLAS_FAMILY        = str(PARCELLATION['atlas']).strip()          # "Craddock" | "Schaefer" | "MIST"
AF_LOWER            = ATLAS_FAMILY.lower()
ATLAS_N_ROIS        = int(PARCELLATION['n_rois'])
ATLAS_NETWORK_SCALE = PARCELLATION.get('network_scale', None)     # <-- used only for Schaefer

atl_cfg = config['atlases']

# Craddock block provides the canonical MNI template label & resolution:
cr_cfg          = atl_cfg['Craddock']
CANONICAL_MNI   = cr_cfg.get('canonical_mni', 'MNI152NLin2009cAsym')
CANONICAL_RES_MM = int(cr_cfg.get('canonical_res', 2))            # <-- used for Craddock harmonization

# Validate vs atlas-specific allowed settings:
if AF_LOWER == "craddock":
    allowed_rois = set(int(x) for x in cr_cfg['allowed_num_ROIs'])
    if ATLAS_N_ROIS not in allowed_rois:
        raise ValueError(
            f"Craddock: n_rois={ATLAS_N_ROIS} not in allowed_num_ROIs={sorted(allowed_rois)}")

    # Craddock does not use network_scale, so we can ignore any provided value:
    if ATLAS_NETWORK_SCALE not in (None, "", "None"):
        print("[INFO] Craddock atlas does not use 'network_scale'; ignoring provided value.")
    ATLAS_NETWORK_SCALE = None

elif AF_LOWER == "schaefer":
    sch_cfg = atl_cfg['Schaefer']
    allowed_scales_map = sch_cfg.get('allowed_scales', {})  # <-- dict: {n_rois: [7, 17], ...}

    # For Schaefer, network_scale is required (7 or 17):
    if ATLAS_NETWORK_SCALE in (None, "", "None"):
        raise ValueError(
            "Schaefer atlas requires 'parcellation.network_scale' (e.g., 7 or 17).")

    ATLAS_NETWORK_SCALE = int(ATLAS_NETWORK_SCALE)

    # Validate n_rois and network_scale combination:
    if ATLAS_N_ROIS not in allowed_scales_map:
        raise ValueError(
            f"Schaefer: n_rois={ATLAS_N_ROIS} not in allowed_scales "
            f"(keys={list(allowed_scales_map.keys())}). "
            "Use one of the allowed keys (e.g., 100, 200, 300, 400).")

    valid_scales = [int(x) for x in allowed_scales_map[ATLAS_N_ROIS]]
    if ATLAS_NETWORK_SCALE not in valid_scales:
        raise ValueError(
            f"Schaefer: network_scale={ATLAS_NETWORK_SCALE} not allowed for n_rois={ATLAS_N_ROIS}. "
            f"Valid options: {valid_scales}")

elif AF_LOWER == "mist":
    mist_cfg = atl_cfg['MIST']
    allowed_rois = [int(x) for x in mist_cfg.get('allowed_num_ROIs', [])]
    if allowed_rois and (ATLAS_N_ROIS not in allowed_rois):
        raise ValueError(
            f"MIST: n_rois={ATLAS_N_ROIS} not in allowed_num_ROIs={allowed_rois}.")

    # MIST uses only n_rois; ignore any provided network_scale
    if ATLAS_NETWORK_SCALE not in (None, "", "None"):
        print("[INFO] MIST atlas ignores 'network_scale'; using n_rois only.")
    ATLAS_NETWORK_SCALE = None

else:
    raise ValueError(f"Unsupported atlas: {ATLAS_FAMILY!r} (expected Craddock, Schaefer, or MIST)")

# Human-readable tag for filenames (must match what parcellation script expects):
if AF_LOWER == "craddock":
    ATLAS_TAG = f"craddock-{ATLAS_N_ROIS:03d}"
elif AF_LOWER == "schaefer":
    ATLAS_TAG = f"schaefer-{ATLAS_N_ROIS:03d}p-{ATLAS_NETWORK_SCALE}net"
else:  # MIST
    ATLAS_TAG = f"mist-{ATLAS_N_ROIS}"

print(
    f"[ATLAS] family={ATLAS_FAMILY}, tag={ATLAS_TAG}, "
    f"n_rois={ATLAS_N_ROIS}, network_scale={ATLAS_NETWORK_SCALE}, "
    f"canonical={CANONICAL_MNI}@{CANONICAL_RES_MM}mm")


# ============================
# PATHS
# ============================

BASE_DIRECTORY        = Path(config['root_output_directory'])

RUN_MANIFEST_PATH     = BASE_DIRECTORY / 'subject_manifest.csv'
fMRI_PARAMETERS_PATH  = BASE_DIRECTORY / 'fMRI_manifest.csv'

BOLDREF_DIR       = BASE_DIRECTORY / config['SDC_output_dir']
DENOISED_DIR      = BASE_DIRECTORY / config['denoising_output_dir']
EPI_BRAINMASK_DIR = BASE_DIRECTORY / config['BBR_output_dir']

ALIGNMENT_PATH    = BASE_DIRECTORY / config['alignment_output_dir']
TRANSFORMS_SUBDIR = config['T1_to_MNI_transforms_subdir']

QC_DIR = BASE_DIRECTORY / config['alignment_QC_subdir']

if isinstance(config['intermediate_QC_subdir'], str) and \
   config['intermediate_QC_subdir'].strip().lower() not in ('none', 'false', 'na', 'nan', ''):
    QC_INTERMEDIATE_DIR = QC_DIR / config['intermediate_QC_subdir']
else:
    QC_INTERMEDIATE_DIR = QC_DIR

CRADDOCK_SRC_DIR = Path(config['atlases']['Craddock']['craddock_dir'])


# ============================
# TEMP HANDLING
# ============================

import tempfile
from tempfile import gettempdir as _gettempdir

TEMP_REDIRECT_VALID = False
if REDIRECT_TEMP_TOGGLE:
    if isinstance(REDIRECT_TEMP_FOLDER, str) and REDIRECT_TEMP_FOLDER.strip().lower() not in ('none', 'off', 'false', 'na', 'nan', ''):
        # Interpret redirect_temp_folder as relative to BASE_DIRECTORY, like before
        TMP_DIR = BASE_DIRECTORY / REDIRECT_TEMP_FOLDER
        TEMP_REDIRECT_VALID = True
    else:
        print("[WARN] redirect_temp_toggle=True but redirect_temp_folder invalid; "
              "using default temp under ALIGNMENT_PATH.")
        TMP_DIR = ALIGNMENT_PATH / "_tmp"
else:
    TMP_DIR = ALIGNMENT_PATH / "_tmp"

TMP_DIR.mkdir(parents=True, exist_ok=True)

os.environ["TMPDIR"] = str(TMP_DIR)
os.environ["TEMP"]   = str(TMP_DIR)
os.environ["TMP"]    = str(TMP_DIR)
tempfile.tempdir = str(TMP_DIR)
try:
    tempfile._tempdir = str(TMP_DIR)  # type: ignore[attr-defined]
except Exception:
    pass

print(f"[TEMP] Using tmp at: {TMP_DIR}")
print(f"[TEMP] tempfile.gettempdir() = {_gettempdir()}")

ATLAS_MNI_PATH = TMP_DIR / f"atlas_{ATLAS_TAG}_mni{CANONICAL_RES_MM}mm.nii.gz"
MNI_REF_PATH   = TMP_DIR / f"mni2009c_{CANONICAL_RES_MM}mm_ref.nii.gz"

ATLAS_T1_FILENAME  = f"atlas_{ATLAS_TAG}_on_T1.nii.gz"
ATLAS_EPI_FILENAME = f"atlas_{ATLAS_TAG}_on_EPI.nii.gz"


# ============================
# LOAD MANIFESTS + SUBSETTING
# ============================

RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
fMRI_runs    = pd.read_csv(fMRI_PARAMETERS_PATH)

In [ ]:
# =====================================================================
# FILTERING (group_ID → session_ID → subject_ID) + DIAGNOSTIC SUBSETTING
# =====================================================================

# ---- FILTERING (if enabled) ----
any_filters_active = bool(FILTER_GROUP_IDS or FILTER_SESSION_IDS or FILTER_SUBJECT_IDS)
if any_filters_active:
    print("\n[FILTER] Applying YAML-defined filters to fMRI_runs...")
    print(f"[FILTER] Starting with {len(fMRI_runs):,} rows.")
    # 1) Filter by group_IDs (substrings, case-insensitive):
    if FILTER_GROUP_IDS:
        n_before = len(fMRI_runs)
        pattern = "|".join(re.escape(val) for val in FILTER_GROUP_IDS)
        mask = fMRI_runs["group_ID"].astype(str).str.contains(pattern, case=False, na=False)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] group_IDs {FILTER_GROUP_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    # 2) Filter by session_IDs (exact matches):
    if FILTER_SESSION_IDS:
        n_before = len(fMRI_runs)
        mask = fMRI_runs["session_ID"].astype(str).isin(FILTER_SESSION_IDS)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] session_IDs {FILTER_SESSION_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    # 3) Filter by subject_IDs (exact matches):
    if FILTER_SUBJECT_IDS:
        n_before = len(fMRI_runs)
        mask = fMRI_runs["subject_ID"].astype(str).isin(FILTER_SUBJECT_IDS)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] subject_IDs {FILTER_SUBJECT_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    print(f"[FILTER] Final row count after all filters: {len(fMRI_runs):,} rows.\n")

# ---- DIAGNOSTIC SUBSETTING (if enabled) ----
if isinstance(SUBSET, int) and SUBSET > 0:
    print(f"\nDIAGNOSTIC SUBSETTING ENABLED: Running only {SUBSET} test subjects/files:")
    fMRI_runs = fMRI_runs.head(SUBSET).copy()

if any_filters_active or SUBSET:
    fMRI_runs = fMRI_runs.reset_index(drop=True)
    display(fMRI_runs)

---------

First, confirm Craddock atlas ROIs (shown in MNI space, pre-transformation):

In [ ]:
# ============================
# CELL 2 — ATLAS LOAD & HARMONIZATION
# ============================

def _is_integer_like(arr, atol=1e-6):
    a = np.asanyarray(arr)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return True
    return np.max(np.abs(a - np.rint(a))) <= atol

def _max_frac_part(arr):
    a = np.asanyarray(arr)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return 0.0
    return float(np.max(np.abs(a - np.rint(a))))

def _coerce_labels_integer(img):
    dat = img.get_fdata()
    if not _is_integer_like(dat):
        dat = np.rint(dat)
    dat = dat.astype(np.int32, copy=False)
    return nib.Nifti1Image(dat, img.affine, img.header)

def load_atlas_mni2mm_in_memory(atlas_path):
    atlas_img = nib.load(atlas_path)
    atlas_int = _coerce_labels_integer(atlas_img)
    mni_ref = load_mni152_template(resolution=CANONICAL_RES_MM)
    atlas_on_mni2mm = resample_to_img(
        atlas_int, mni_ref, interpolation="nearest", fill_value=0)
    data_i32 = np.rint(atlas_on_mni2mm.get_fdata()).astype(np.int32, copy=False)
    atlas_on_mni2mm = nib.Nifti1Image(data_i32, atlas_on_mni2mm.affine, atlas_on_mni2mm.header)
    return atlas_on_mni2mm, mni_ref

# --- Resolve source atlas in MNI space (family-specific) ---

if AF_LOWER == "craddock":
    # Local Craddock maps: expect something like "Craddock-50_ROIs.nii"
    brain_map_path = CRADDOCK_SRC_DIR / f"Craddock-{ATLAS_N_ROIS}_ROIs.nii"
    if not brain_map_path.exists():
        raise FileNotFoundError(f"Craddock atlas not found at {brain_map_path}")
    family_desc = f"Craddock {ATLAS_N_ROIS}-ROI atlas"

elif AF_LOWER == "schaefer":
    # Use nilearn to fetch (download once; then cached)
    print(f"[ATLAS] Fetching Schaefer-2018 atlas via nilearn: n_rois={ATLAS_N_ROIS}, "
          f"yeo_networks={ATLAS_NETWORK_SCALE}, res={CANONICAL_RES_MM}mm")
    sch = datasets.fetch_atlas_schaefer_2018(
        n_rois=ATLAS_N_ROIS,
        yeo_networks=ATLAS_NETWORK_SCALE,
        resolution_mm=CANONICAL_RES_MM)
    brain_map_path = Path(sch.maps)
    family_desc = f"Schaefer-2018 {ATLAS_N_ROIS}-parcel, {ATLAS_NETWORK_SCALE}-network atlas"

elif AF_LOWER == "mist":
    # Use nilearn BASC / MIST multiscale atlas
    print(f"[ATLAS] Fetching MIST/BASC multiscale atlas via nilearn: scale={ATLAS_N_ROIS}")
    mist = datasets.fetch_atlas_basc_multiscale_2015()
    key = f"scale{ATLAS_N_ROIS:03d}"
    if not hasattr(mist, key):
        raise RuntimeError(
            f"MIST atlas fetcher did not return attribute {key}. "
            f"Available: {[k for k in mist.keys() if k.startswith('scale')]}")
    brain_map_path = Path(getattr(mist, key))
    family_desc = f"MIST/BASC multiscale atlas @ {ATLAS_N_ROIS} parcels"

else:
    raise RuntimeError(f"Unexpected atlas family: {ATLAS_FAMILY!r}")


# --- Inspect original atlas metadata ---

brain_map = nib.load(str(brain_map_path))

print("=== Original atlas metadata ===")
print(f"Family: {family_desc}")
print(f"Path:   {brain_map_path}")
print(f"Shape:  {brain_map.shape}")
print(f"Voxel size (mm): {brain_map.header.get_zooms()[:3]}")
print(f"dtype (on-disk): {brain_map.get_data_dtype()}")

orig_sample = np.asanyarray(brain_map.get_fdata(dtype=np.float32))
integer_like = _is_integer_like(orig_sample)
max_frac = _max_frac_part(orig_sample)
print(f"Integer-like: {integer_like}  [max |fractional part|={max_frac:.2e}]")
uniq = np.unique(orig_sample.astype(np.int64) if integer_like else np.rint(orig_sample).astype(np.int64))
print(f"Approx. unique labels (rounded if needed): {uniq.size} (first 20: {uniq[:20]})")


# --- Harmonize to canonical MNI template @ CANONICAL_RES_MM ---

atlas_mni2mm, mni_template = load_atlas_mni2mm_in_memory(str(brain_map_path))

print("\n=== Harmonized atlas (in memory) ===")
print(f"Template: {CANONICAL_MNI}, {CANONICAL_RES_MM} mm")
print(f"Shape:    {atlas_mni2mm.shape}")
print(f"Voxel size (mm): {atlas_mni2mm.header.get_zooms()[:3]}")
dat_h = atlas_mni2mm.get_fdata()
labels_h = np.unique(dat_h.astype(np.int64))
print(f"Labels: count={labels_h.size}, min={labels_h.min()}, max={labels_h.max()} (0=background)")
print("Note: nearest-neighbor resampling used; labels preserved as integers.\n")

# --- Write/refresh cache files in TMP_DIR (used by downstream cells) ---

need_atlas = OVERWRITE_ATLAS_CACHE or (not ATLAS_MNI_PATH.exists())
need_ref   = OVERWRITE_ATLAS_CACHE or (not MNI_REF_PATH.exists())

if need_atlas:
    dat_i16 = dat_h.astype(np.int16, copy=False)
    nib.save(nib.Nifti1Image(dat_i16, atlas_mni2mm.affine, atlas_mni2mm.header), str(ATLAS_MNI_PATH))
    print(f"[CACHE] wrote harmonized atlas → {ATLAS_MNI_PATH}")
else:
    print(f"[CACHE] using existing harmonized atlas: {ATLAS_MNI_PATH}")

if need_ref:
    nib.save(mni_template, str(MNI_REF_PATH))
    print(f"[CACHE] wrote MNI ref → {MNI_REF_PATH}")
else:
    print(f"[CACHE] using existing MNI ref: {MNI_REF_PATH}")

# Quick overlay plot on MNI template (visual sanity check):
plotting.plot_roi(
    str(ATLAS_MNI_PATH),  # cached file
    bg_img=str(MNI_REF_PATH),
    title=f"{ATLAS_TAG} (harmonized) over {CANONICAL_MNI} {CANONICAL_RES_MM}mm",
    display_mode="ortho",
    draw_cross=False)
plt.show()

-------

Next, we construct a full list of candidate subjects (based on subdirectories present in the 'denoising' folder, from the previous pipeline step), and also audit each subject/file for the necessary data inputs we'll need to proceed with parcellation:

In [ ]:
# === CELL 3 — Build + audit filetargets_df from fMRI_runs ===

# Sanity-check required globals and directories
for name, path in [
    ("DENOISED_DIR", DENOISED_DIR),
    ("BOLDREF_DIR", BOLDREF_DIR),
    ("EPI_BRAINMASK_DIR", EPI_BRAINMASK_DIR),
    ("ALIGNMENT_PATH", ALIGNMENT_PATH)]:
    if path is None:
        raise RuntimeError(f"[INIT] Global {name} is None; check init cell.")
    if not Path(path).exists():
        raise RuntimeError(f"[INIT] {name} does not exist on disk: {path}")

# Required columns in fMRI_runs
required_cols = ["subject_ID", "session_ID"]
for col in required_cols:
    if col not in fMRI_runs.columns:
        raise RuntimeError(f"[INIT] fMRI_runs is missing required column '{col}'")

# Helper: construct prefix for this script (subject_session only)
def _make_prefix(subject_id: str, session_id: str) -> str:
    return f"{subject_id}_{session_id}"

# Helper: robust boldref lookup (accept .nii.gz or .nii)
def _boldref_path(prefix: str) -> Optional[Path]:
    """
    Return the boldref image for this prefix, preferring .nii.gz if present,
    otherwise falling back to .nii. Returns a Path or None if neither exists.
    """
    p_niigz = BOLDREF_DIR / prefix / "boldref_sdc.nii.gz"
    p_nii   = BOLDREF_DIR / prefix / "boldref_sdc.nii"

    if p_niigz.exists():
        return p_niigz
    if p_nii.exists():
        return p_nii
    return None

problems = []
rows = []

for _, row in fMRI_runs.iterrows():
    subj = str(row["subject_ID"])
    sess = str(row["session_ID"])
    grp  = str(row.get("group_ID", "NA"))  # group_ID may or may not exist

    prefix = _make_prefix(subj, sess)

    # Core inputs for this script:
    denoised_p       = DENOISED_DIR      / prefix / "bold_denoised.nii.gz"
    boldref_p        = _boldref_path(prefix)
    brainmask_p      = EPI_BRAINMASK_DIR / prefix / "brainmask_epi.nii.gz"
    gm_p             = EPI_BRAINMASK_DIR / prefix / "gm_epi.nii.gz"
    t1_to_epi_itk_p  = EPI_BRAINMASK_DIR / prefix / "t1toepi_itk.txt"

    missing_inputs = []
    if not denoised_p.exists():
        missing_inputs.append("bold_denoised.nii.gz")

    # NEW: accept either boldref_sdc.nii.gz or boldref_sdc.nii
    if boldref_p is None:
        missing_inputs.append("boldref_sdc (nii / nii.gz)")

    if not brainmask_p.exists():
        missing_inputs.append("brainmask_epi.nii.gz")

    if not gm_p.exists():
        missing_inputs.append("gm_epi.nii.gz")

    if not t1_to_epi_itk_p.exists():
        missing_inputs.append("t1toepi_itk.txt")

    if missing_inputs:
        problems.append({
            "prefix": prefix,
            "issue": "missing inputs",
            "detail": ", ".join(missing_inputs)})

    rows.append({
        "group_ID": grp,
        "subject_ID": subj,
        "session_ID": sess,
        "prefix": prefix,
        "bold_denoised": str(denoised_p) if denoised_p.exists() else "",
        "boldref_sdc": str(boldref_p) if boldref_p is not None else "",
        "brainmask_epi": str(brainmask_p) if brainmask_p.exists() else "",
        "gm_epi": str(gm_p) if gm_p.exists() else "",
        "t1_to_epi_itk": str(t1_to_epi_itk_p) if t1_to_epi_itk_p.exists() else "",
        "missing": ", ".join(missing_inputs) if missing_inputs else ""})

filetargets_df = (
    pd.DataFrame(rows)
    .sort_values(["subject_ID", "session_ID"])
    .reset_index(drop=True))

print(f"[AUDIT] Built filetargets_df with {len(filetargets_df)} rows.")

if problems:
    problems_df = pd.DataFrame(problems).sort_values("prefix").reset_index(drop=True)
    print(f"[AUDIT] {len(problems_df)} rows have missing prerequisites:")
    with pd.option_context("display.max_rows", 50, "display.max_colwidth", 120):
        display(problems_df)

    if HARD_STOP:
        raise RuntimeError(
            f"[HARD_STOP] Alignment script: found {len(problems_df)} rows with missing prerequisites. "
            "See problems_df above.")
    else:
        bad_prefixes = set(problems_df["prefix"])
        before = len(filetargets_df)
        filetargets_df = (
            filetargets_df[~filetargets_df["prefix"].isin(bad_prefixes)]
            .reset_index(drop=True))
        after = len(filetargets_df)
        print(f"[AUDIT] Dropped {before - after} problematic rows; continuing with {after} rows.")
else:
    print("[AUDIT] All rows passed prerequisite checks; no problems detected.")

with pd.option_context("display.max_rows", 10, "display.max_colwidth", 140):
    display(filetargets_df.head())

Next, we generate T1 --> MNI transforms

In [ ]:
# === CELL 4 — Compute / cache T1→MNI transforms (0GenericAffine, 1Warp, 1InverseWarp) ===
#   - Writes ONE canonical set of transforms per subject_ID/session/prefix:
#       <ALIGNMENT_PATH>/<prefix>/<TRANSFORMS_SUBDIR>/
#         0GenericAffine.mat
#         1Warp.nii.gz
#         1InverseWarp.nii.gz
#   - Registration uses the SAME T1 representation we'll later apply to a temp NIfTI in ALIGNMENT_PATH/_tmp

# Ensure required columns exist (create t1_mni_* columns if missing)
required_cols = [
    "subject_ID", "session_ID", "prefix",
    "bold_denoised", "boldref_sdc",
    "brainmask_epi", "gm_epi", "t1_to_epi_itk",
    "t1_mni_affine", "t1_mni_warp", "t1_mni_invwarp",
    "missing"]

for col in ("t1_mni_affine", "t1_mni_warp", "t1_mni_invwarp"):
    if col not in filetargets_df.columns:
        filetargets_df[col] = "missing"

for col in required_cols:
    if col not in filetargets_df.columns:
        raise RuntimeError(f"filetargets_df is missing column: {col}")

# Normalize empty T1→MNI values to "missing":
for col in ("t1_mni_affine", "t1_mni_warp", "t1_mni_invwarp"):
    filetargets_df[col] = (
        filetargets_df[col]
        .fillna("missing")
        .replace("", "missing"))

# Paths / constants:
OUT_T1MNI_DIR = ALIGNMENT_PATH          # <-- root alignment dir from init cell
OUT_T1MNI_DIR.mkdir(parents=True, exist_ok=True)

# Use the TMP_DIR already defined in the init cell; if not present, create ALIGNMENT_PATH/_tmp:
if "TMP_DIR" not in globals():
    TMP_DIR = ALIGNMENT_PATH / "_tmp"
TMP_DIR.mkdir(parents=True, exist_ok=True)

FS_SUBJECTS_DIR = Path(FREESURFER_SUBJECTS_DIR)
if not FS_SUBJECTS_DIR.exists():
    raise RuntimeError(f"FREESURFER_SUBJECTS_DIR does not exist: {FS_SUBJECTS_DIR}")

def load_fs_t1_to_tmp(subject_id: str) -> Path:
    """
    Prefer FreeSurfer brain.mgz, then T1.mgz from FREESURFER_SUBJECTS_DIR;
    convert to a temp NIfTI in TMP_DIR. This exact NIfTI grid must be reused
    when applying transforms.
    """
    candidates = [
        FS_SUBJECTS_DIR / subject_id / "mri" / "brain.mgz",
        FS_SUBJECTS_DIR / subject_id / "mri" / "T1.mgz"]
    for mgz in candidates:
        if mgz.exists():
            img = nib.load(str(mgz))
            out = TMP_DIR / f"{subject_id}_brain_for_ants.nii.gz"
            nib.save(
                nib.Nifti1Image(
                    img.get_fdata().astype(np.float32),
                    img.affine),
                str(out))
            return out
    raise FileNotFoundError(
        f"FreeSurfer T1 not found for subject '{subject_id}'. "
        f"Searched brain.mgz/T1.mgz under {FS_SUBJECTS_DIR / subject_id / 'mri'}")

def nifti_to_ants(path: Path):
    import ants
    return ants.image_read(str(path))

def describe_ants(img):
    return {
        "shape": tuple(int(x) for x in img.shape),
        "spacing": tuple(float(x) for x in img.spacing),
        "origin": tuple(float(x) for x in img.origin),
        "direction_00": float(img.direction[0][0]) if hasattr(img, "direction") else None}

# _________________________________________________________________________________
# Main execution: build MNI ref/mask and compute T1→MNI where needed:

try:
    import ants
except Exception as e:
    raise RuntimeError(f"ANTsPy is required for this cell: {e}")

# Prepare fixed MNI + mask once (2 mm MNI from nilearn)
mni_ref_img  = load_mni152_template(resolution=CANONICAL_RES_MM)
mni_masker   = NiftiMasker(standardize=False)
mni_masker.fit(mni_ref_img)

mni_mask_img = nib.Nifti1Image(
    mni_masker.mask_img_.get_fdata().astype(np.uint8),
    mni_ref_img.affine,
    mni_ref_img.header)

tmp_mni_ref  = TMP_DIR / "_tmp_mni_ref.nii.gz"
tmp_mni_mask = TMP_DIR / "_tmp_mni_mask.nii.gz"
nib.save(mni_ref_img,  str(tmp_mni_ref))
nib.save(mni_mask_img, str(tmp_mni_mask))

ants_fixed = ants.image_read(str(tmp_mni_ref))
ants_fmask = ants.image_read(str(tmp_mni_mask))

print(f"[MNI ] shape={ants_fixed.shape} spacing={ants_fixed.spacing} origin={ants_fixed.origin}")

# Determine which rows need T1→MNI transforms
if OVERWRITE_TRANSFORMS:
    todo_idx = list(filetargets_df.index)
    print(f"OVERWRITE_TRANSFORMS=True -> recomputing T1→MNI for ALL {len(todo_idx)} rows")
else:
    todo_idx = filetargets_df.index[filetargets_df["t1_mni_invwarp"] == "missing"].tolist()
    print(f"Subjects needing T1→MNI (invwarp == 'missing'): {len(todo_idx)}")

for i in todo_idx:
    prefix     = str(filetargets_df.at[i, "prefix"])
    subject_id = str(filetargets_df.at[i, "subject_ID"])

    # Per-subject transforms live at ALIGNMENT_PATH/<prefix>/<TRANSFORMS_SUBDIR>/
    out_dir = OUT_T1MNI_DIR / prefix / TRANSFORMS_SUBDIR
    out_dir.mkdir(parents=True, exist_ok=True)

    # Canonical output names:
    out_aff   = out_dir / "0GenericAffine.mat"
    out_warp  = out_dir / "1Warp.nii.gz"
    out_iwarp = out_dir / "1InverseWarp.nii.gz"

    have_all = out_aff.exists() and out_warp.exists() and out_iwarp.exists()
    if have_all and (not OVERWRITE_TRANSFORMS):
        filetargets_df.at[i, "t1_mni_affine"]  = str(out_aff)
        filetargets_df.at[i, "t1_mni_warp"]    = str(out_warp)
        filetargets_df.at[i, "t1_mni_invwarp"] = str(out_iwarp)
        print(f"[USE ] {prefix}: already present (aff/warp/invwarp).")
        continue

    # Load T1 as the temp NIfTI we’ll reuse later:
    try:
        t1_tmp_nii = load_fs_t1_to_tmp(subject_id)
        ants_moving = nifti_to_ants(t1_tmp_nii)
    except Exception as e:
        print(f"[SKIP] {prefix}: failed to load/prepare T1 temp NIfTI: {e}")
        continue

    desc_t1 = describe_ants(ants_moving)
    print(
        f"[T1  ] {subject_id} path={t1_tmp_nii.name} "
        f"shape={desc_t1['shape']} spacing={desc_t1['spacing']} origin={desc_t1['origin']}")

    # Run registration; copy canonical outputs to our target names:
    print(f"[REG ] {prefix}: ants.registration(type={REG_TYPE}) …")
    reg = ants.registration(
        fixed=ants_fixed,
        moving=ants_moving,
        type_of_transform=REG_TYPE,
        mask=ants_fmask,
        verbose=False)
    fwd = list(reg["fwdtransforms"])
    inv = list(reg["invtransforms"])

    # Locate sources; copy to canonical names:
    src_aff  = next(p for p in fwd if p.endswith(".mat"))
    src_warp = next(p for p in fwd if p.endswith(".gz"))
    src_invw = next(p for p in inv if p.endswith(".gz"))

    shutil.copy2(src_aff,  str(out_aff))
    shutil.copy2(src_warp, str(out_warp))
    shutil.copy2(src_invw, str(out_iwarp))

    # Sanity-check -- file sizes (nonzero and plausibly large for warps):
    sz_aff  = out_aff.stat().st_size
    sz_wrp  = out_warp.stat().st_size
    sz_iwrp = out_iwarp.stat().st_size
    print(f"[OUT ] {prefix}: aff={sz_aff}B warp={sz_wrp}B invwarp={sz_iwrp}B")
    if sz_wrp < 1_000_000 or sz_iwrp < 1_000_000:
        print(f"[WARN] {prefix}: warp files are unexpectedly small. "
              "Downstream apply may fail; verify T1 grid consistency.")

    # Update table and write provenance:
    filetargets_df.at[i, "t1_mni_affine"]  = str(out_aff)
    filetargets_df.at[i, "t1_mni_warp"]    = str(out_warp)
    filetargets_df.at[i, "t1_mni_invwarp"] = str(out_iwarp)

    prov = {
        "prefix": prefix,
        "fs_subject_ID": subject_id,
        "registration": {
            "type_of_transform": REG_TYPE,
            "fixed_template": "MNI152NLin2009cAsym 2mm (nilearn)",
            "moving_t1_tmp_nii": str(t1_tmp_nii),
            "moving_t1_geometry": desc_t1,
            "mni_geometry": describe_ants(ants_fixed)},
        "outputs": {
            "affine": str(out_aff),
            "warp": str(out_warp),
            "invwarp": str(out_iwarp)}}
    with open(out_dir / f"{prefix}_provenance.json", "w") as f:
        json.dump(prov, f, indent=2)

remaining = int((filetargets_df["t1_mni_invwarp"] == "missing").sum())
print(f"Remaining subjects missing T1→MNI: {remaining}")

In [ ]:
# ============================
# CELL 5 — MNI→T1→EPI ATLAS PROJECTION + QC
# ============================

# Variant selection & QC toggles:
VARIANT_SELECTION   = "t1mask"   # <-- "nonzero" or "t1mask"
PRINT_EPI_COVERAGE  = True       # <-- no reason not to keep this always enabled IMO
EPI_QC_SCALE_UP     = 1.75       # <-- set to 1.0 to disable scaling

# Sanity-check harmonized atlas + ref:
if (not ATLAS_MNI_PATH.exists()) or (not MNI_REF_PATH.exists()):
    have = sorted([p.name for p in TMP_DIR.glob("*")])
    raise RuntimeError(
        "Expected harmonized atlas/ref not found. Run the 'atlas load & harmonization' cell first.\n"
        f"Looked for:\n  {ATLAS_MNI_PATH}\n  {MNI_REF_PATH}\n"
        f"_tmp contains: {have}")

try:
    import ants
except Exception as e:
    raise RuntimeError(f"ANTsPy is required for this cell: {e}")

def load_t1_brainmask(subject_id: str, ants_t1) -> "ants.core.ants_image.ANTsImage":
    """
    Try FreeSurfer T1-space mask first; else derive via ANTs.
    """
    fs_mask_candidates = [
        Path(os.environ["SUBJECTS_DIR"]) / subject_id / "mri" / "brainmask.mgz",
        Path(os.environ["SUBJECTS_DIR"]) / subject_id / "mri" / "brain.finalsurfs.mgz"]
    for mgz in fs_mask_candidates:
        if mgz.exists():
            m = nib.load(str(mgz))
            tmp_mask = TMP_DIR / f"{subject_id}_fs_brainmask_tmp.nii.gz"
            nib.save(nib.Nifti1Image((m.get_fdata() > 0).astype(np.uint8), m.affine), str(tmp_mask))
            mask_ants = ants.image_read(str(tmp_mask))
            if mask_ants.shape != ants_t1.shape or mask_ants.spacing != ants_t1.spacing:
                mask_ants = ants.resample_image_to_target(mask_ants, ants_t1, interp="nearestNeighbor")
            return mask_ants
    try:
        return ants.get_mask(ants_t1)
    except Exception:
        thr = ants.threshold_image(
            ants_t1,
            lower=ants_t1.quantile(0.25),
            upper=ants_t1.max(),
            inside_value=1,
            outside_value=0)
        return thr

def nonzero_stats_ants(ants_img):
    arr = ants_img.numpy()
    nz  = int((arr != 0).sum())
    return nz, float(arr.min(initial=0.0)), float(arr.max(initial=0.0))

def save_as_int16(ants_img, out_path):
    arr = ants_img.numpy().astype(np.int16, copy=False)
    out = ants.from_numpy(arr)
    out = ants.copy_image_info(ants_img, out)
    ants.image_write(out, str(out_path))

def _scale_up_png_inplace(png_path: Path, scale: float):
    if scale is None or scale <= 1.0:
        return
    try:
        from PIL import Image
        im = Image.open(str(png_path))
        w, h = im.size
        im = im.resize((int(w*scale), int(h*scale)), resample=Image.NEAREST)
        im.save(str(png_path))
    except Exception as e:
        print(f"[WARNING] !!! Could not scale up QC PNG ({png_path.name}): {e}")

def qc_overlay_roi(bg_path, roi_path, out_png, title):
    try:
        plotting.plot_roi(
            roi_path, bg_img=bg_path,
            display_mode="ortho", draw_cross=False,
            title=title, cmap="nipy_spectral", alpha=0.5,
            output_file=str(out_png))
    except Exception as e:
        print(f"[WARNING] !!! QC overlay (nilearn) failed ({title}): {e}")
    if not Path(out_png).exists():
        # Fallback -- simple mid-slice overlay:
        try:
            bg = nib.load(bg_path).get_fdata()
            roi = nib.load(roi_path).get_fdata()
            cx, cy, cz = [s//2 for s in bg.shape[:3]]
            fig, axes = plt.subplots(1, 3, figsize=(11, 3.5)); axes = axes.ravel()
            axes[0].imshow(bg[:, :, cz].T, origin='lower'); axes[0].imshow(np.ma.masked_equal(roi[:, :, cz], 0).T, origin='lower', alpha=0.5)
            axes[1].imshow(bg[:, cy, :].T, origin='lower'); axes[1].imshow(np.ma.masked_equal(roi[:, cy, :], 0).T, origin='lower', alpha=0.5)
            axes[2].imshow(bg[cx, :, :].T, origin='lower'); axes[2].imshow(np.ma.masked_equal(roi[cx, :, :], 0).T, origin='lower', alpha=0.5)
            for ax in axes: ax.axis('off')
            fig.suptitle(title, y=0.98); fig.tight_layout()
            fig.savefig(str(out_png), dpi=120); plt.close(fig)
        except Exception as e:
            print(f"[WARNING] !!! QC overlay (fallback) failed ({title}): {e}")

def project_atlas_mni_to_t1_both(ants_t1, aff_path, invwarp_path, ants_atlas_mni):
    """
    Compute both inverse orderings:
      A: [invwarp, affine] with whichtoinvert=[False, True]
      B: [affine, invwarp] with whichtoinvert=[True,  False]
    """
    A = ants.apply_transforms(
        fixed=ants_t1, moving=ants_atlas_mni,
        transformlist=[str(invwarp_path), str(aff_path)],
        whichtoinvert=[False, True], interpolator="nearestNeighbor")
    nzA, _, _ = nonzero_stats_ants(A)

    B = ants.apply_transforms(
        fixed=ants_t1, moving=ants_atlas_mni,
        transformlist=[str(aff_path), str(invwarp_path)],
        whichtoinvert=[True, False], interpolator="nearestNeighbor")
    nzB, _, _ = nonzero_stats_ants(B)

    return A, B, nzA, nzB

def choose_variant_by_t1mask(prefix, A, B, nzA, nzB, t1_mask):
    aA = (A.numpy() != 0)
    aB = (B.numpy() != 0)
    m  = (t1_mask.numpy() > 0)

    inA  = int((aA & m).sum()); outA = int((aA & (~m)).sum())
    inB  = int((aB & m).sum()); outB = int((aB & (~m)).sum())
    delta_pct = 100.0 * abs(nzA - nzB) / max(nzA, nzB) if max(nzA, nzB) > 0 else 0.0

    if inB > inA:
        chosen = "B"
    elif inA > inB:
        chosen = "A"
    else:
        if outA < outB:
            chosen = "A"
        elif outB < outA:
            chosen = "B"
        else:
            chosen = "A" if nzA >= nzB else "B"

    print(f"[VARIANT] {prefix}: nzA={nzA} nzB={nzB} (Δ={delta_pct:.1f}%) | "
          f"inA={inA} outA={outA} | inB={inB} outB={outB} -> chosen={chosen}")
    return chosen

def describe_ants(img):
    return {
        "shape": tuple(int(x) for x in img.shape),
        "spacing": tuple(float(x) for x in img.spacing),
        "origin": tuple(float(x) for x in img.origin),
        "direction_00": float(img.direction[0][0]) if hasattr(img, "direction") else None}

# Load atlas & ref once:
ants_atlas_mni = ants.image_read(str(ATLAS_MNI_PATH))
ants_mni_ref   = ants.image_read(str(MNI_REF_PATH))
print(f"[ATLAS] {ATLAS_MNI_PATH.name} shape={ants_atlas_mni.shape} spacing={ants_atlas_mni.spacing}")
print(f"[MNI  ] {MNI_REF_PATH.name} shape={ants_mni_ref.shape} spacing={ants_mni_ref.spacing}")

wrote = 0
failed = 0
inspected = 0
variant_disagreements = 0

for _, row in filetargets_df.iterrows():
    prefix = str(row["prefix"])
    subj   = str(row["subject_ID"])

    boldref = Path(row["boldref_sdc"]) if row["boldref_sdc"] else None
    t1_to_epi_itk = Path(row["t1_to_epi_itk"]) if row["t1_to_epi_itk"] else None

    if not boldref or not boldref.exists() or not t1_to_epi_itk or not t1_to_epi_itk.exists():
        print(f"[SKIP] {prefix}: missing boldref or T1→EPI ITK affine.")
        failed += 1
        continue

    # T1 temp NIfTI (same grid as registration):
    try:
        t1_tmp_nii = TMP_DIR / f"{subj}_brain_for_ants.nii.gz"
        if not t1_tmp_nii.exists():
            # fallback -- regenerate if cell 4 was skipped:
            t1_tmp_nii = load_fs_t1_to_tmp(subj)
        ants_t1    = ants.image_read(str(t1_tmp_nii))
    except Exception as e:
        print(f"[SKIP] {prefix}: failed to load/prepare T1 temp NIfTI: {e}")
        failed += 1
        continue

    tdir   = ALIGNMENT_PATH / prefix / TRANSFORMS_SUBDIR
    aff_p  = tdir / "0GenericAffine.mat"
    inv_p  = tdir / "1InverseWarp.nii.gz"
    warp_p = tdir / "1Warp.nii.gz"  # <-- incl. for completeness

    missing = [p.name for p in (aff_p, inv_p, warp_p) if not p.exists()]
    if missing:
        print(f"[SKIP] {prefix}: missing transforms ({', '.join(missing)}). Did you run the registration cell?")
        failed += 1
        continue

    d_t1  = describe_ants(ants_t1)
    print(f"[T1  ] {subj} path={t1_tmp_nii.name} shape={d_t1['shape']} "
          f"spacing={d_t1['spacing']} origin={d_t1['origin']}")

    inspected += 1

    # Step 1 -- MNI → T1 transform (compute both variants):
    try:
        A, B, nzA, nzB = project_atlas_mni_to_t1_both(ants_t1, aff_p, inv_p, ants_atlas_mni)

        nonzero_choice = "A" if nzA >= nzB else "B"
        t1_mask = load_t1_brainmask(subj, ants_t1)
        t1mask_choice = choose_variant_by_t1mask(prefix, A, B, nzA, nzB, t1_mask)

        if nonzero_choice != t1mask_choice:
            print("      --> WARNING: Inconsistent overlap metrics (nonzero vs T1-mask)")
            variant_disagreements += 1

        chosen = t1mask_choice if VARIANT_SELECTION == "t1mask" else nonzero_choice

        atlas_t1 = A if chosen == "A" else B
        nz_t1, t1_min, t1_max = nonzero_stats_ants(atlas_t1)
        if nz_t1 == 0:
            print(f"[FAIL] {prefix}: MNI→T1 produced empty labels (variant={chosen}).")
            failed += 1
            continue
    except Exception as e:
        print(f"[FAIL] {prefix}: MNI→T1 error: {e}")
        failed += 1
        continue

    # Step 1b — optional T1-space QC:
    out_t1_dir = ALIGNMENT_PATH / prefix
    out_t1_dir.mkdir(parents=True, exist_ok=True)
    t1_qc_png  = QC_INTERMEDIATE_DIR / f"{prefix}_{ATLAS_TAG}_atlas_on_t1.png"
    t1_qc_png.parent.mkdir(parents=True, exist_ok=True)

    tmp_t1_nifti = out_t1_dir / "_tmp_atlas_t1.nii.gz"
    try:
        save_as_int16(atlas_t1, tmp_t1_nifti)
        if SAVE_INTERMEDIATE_QC:
            qc_overlay_roi(
                str(t1_tmp_nii),
                str(tmp_t1_nifti),
                str(t1_qc_png),
                title=f"{prefix} {ATLAS_TAG} atlas on T1 (variant {chosen})")
    except Exception as e:
        print(f"[WARNING] !!! {prefix}: T1 QC overlay failed: {e}")
    finally:
        try:
            tmp_t1_nifti.unlink()
        except Exception:
            pass

    if SAVE_T1_ATLAS:
        save_as_int16(atlas_t1, (out_t1_dir / ATLAS_T1_FILENAME))

    # Step 2 — T1 → EPI via ITK affine:
    try:
        ants_epi_ref = ants.image_read(str(boldref))
        atlas_epi = ants.apply_transforms(
            fixed=ants_epi_ref, moving=atlas_t1,
            transformlist=[str(t1_to_epi_itk)],
            interpolator="nearestNeighbor")
        nz_epi, e_min, e_max = nonzero_stats_ants(atlas_epi)
        if nz_epi == 0:
            print(f"[FAIL] {prefix}: T1→EPI produced empty labels.")
            failed += 1
            continue
    except Exception as e:
        print(f"[FAIL] {prefix}: T1→EPI error: {e}")
        failed += 1
        continue

    # Save EPI atlas + QC:
    out_dir = ALIGNMENT_PATH / prefix
    out_dir.mkdir(parents=True, exist_ok=True)
    out_epi = out_dir / ATLAS_EPI_FILENAME
    save_as_int16(atlas_epi, out_epi)

    epi_qc_png = QC_DIR / f"{prefix}_{ATLAS_TAG}_atlas_on_epi_boldref.png"
    epi_qc_png.parent.mkdir(parents=True, exist_ok=True)

    try:
        tmp_qc_img = out_dir / "_tmp_atlas_epi_for_qc.nii.gz"
        save_as_int16(atlas_epi, tmp_qc_img)
        qc_overlay_roi(
            str(boldref),
            str(tmp_qc_img),
            str(epi_qc_png),
            title=f"{prefix}: {ATLAS_TAG} atlas on EPI (boldref)")
        try:
            tmp_qc_img.unlink()
        except Exception:
            pass

        _scale_up_png_inplace(epi_qc_png, EPI_QC_SCALE_UP)
    except Exception as e:
        print(f"[WARN] {prefix}: QC overlay (boldref) failed: {e}")

    # Optional EPI coverage QC:
    if PRINT_EPI_COVERAGE:
        try:
            t1_mask_for_epi = load_t1_brainmask(subj, ants_t1)
            epi_mask = ants.apply_transforms(
                fixed=ants_epi_ref, moving=t1_mask_for_epi,
                transformlist=[str(t1_to_epi_itk)],
                interpolator="nearestNeighbor")
            ae = (atlas_epi.numpy() != 0)
            me = (epi_mask.numpy()  >  0)
            inE  = int((ae & me).sum()); outE = int((ae & (~me)).sum())
            print(f"[EPI-COV] {prefix}: in={inE} out={outE} (chosen={chosen})")
        except Exception as e:
            print(f"[WARN] {prefix}: EPI coverage metric failed: {e}")

    print(f"[DONE] {prefix}: variant={chosen} | T1 nonzero={nz_t1} | EPI nonzero={nz_epi} | "
          f"range={e_min:.1f}/{e_max:.1f} | EPI atlas={out_epi} | T1 QC={t1_qc_png} | EPI QC={epi_qc_png}")

    wrote += 1

summary = (f"\nDone. EPI atlases written: {wrote} | failed: {failed} | inspected: {inspected}")
if variant_disagreements > 0:
    summary += f" | variant metric disagreements: {variant_disagreements}"
print(summary)

**Final sanity-check:** Ensure that every subject's "denoised" file (i.e. the fMRI file that has been motion-corrected, de-warped, and de-noised in series) exactly matches the coordinate space described by their BOLDREF file. This is crucial because the denoised file is the ultimate target of parcellation & time-series extraction, but the alignment procedure uses the simpler 3D (i.e. "static") BOLDREF file to perform alignment; so if these are not the same (they should be), then parcellation will not be accurate.

In [ ]:
# ============================
# CELL 6 — VERIFY EPI GRID MATCHES DENOISED BOLD
# ============================
# FINAL SANITY CHECK: EPI atlas grid vs denoised BOLD grid ===
# Compare:
#   - ALIGNMENT_PATH/<file_tag>/atlas_craddock_on_EPI.nii.gz
#   - DENOISED_DIR/<file_tag>/bold_denoised.nii.gz
# for shape, voxel size, and affine equality.

mismatches = []
checked = 0
skipped = 0

for _, row in filetargets_df.iterrows():
    prefix = str(row["prefix"])
    den   = Path(row["bold_denoised"]) if row["bold_denoised"] else None
    atl   = ALIGNMENT_PATH / prefix / ATLAS_EPI_FILENAME

    if (den is None) or (not den.exists()) or (not atl.exists()):
        skipped += 1
        continue

    den_img = nib.load(str(den))
    atl_img = nib.load(str(atl))

    same_shape  = den_img.shape[:3] == atl_img.shape
    same_zooms  = np.allclose(den_img.header.get_zooms()[:3], atl_img.header.get_zooms()[:3], atol=1e-6)
    same_affine = np.allclose(den_img.affine, atl_img.affine, atol=1e-5)

    checked += 1
    if not (same_shape and same_zooms and same_affine):
        mismatches.append({
            "prefix": prefix,
            "atlas_shape": atl_img.shape,
            "den_shape3": den_img.shape[:3],
            "atlas_zooms": atl_img.header.get_zooms()[:3],
            "den_zooms": den_img.header.get_zooms()[:3],
            "affine_equal": bool(same_affine)})

if mismatches:
    print(f"[VERIFY] Detected {len(mismatches)} grid mismatches (checked={checked}, skipped={skipped}). Details:")
    for m in mismatches:
        print(f"  [MISMATCH] {m['prefix']}: "
              f"shape {m['atlas_shape']} vs {m['den_shape3']}, "
              f"zooms {m['atlas_zooms']} vs {m['den_zooms']}, "
              f"affine_equal={m['affine_equal']}")
else:
    if checked > 0:
        print(f"[VERIFY] All checked atlases match denoised grids (checked={checked}, skipped={skipped}).")
    else:
        print(f"[VERIFY] No atlases to check yet (checked=0, skipped={skipped}).")

Final cleanup of temp folder (only if a custom / "redirected" temp folder was used):

In [ ]:
# ============================
# CELL 7 — OPTIONAL TEMP CLEANUP
# ============================
# OPTIONAL CELL — Final cleanup of temporary directory used for ANTs / scratch outputs


tmp_root = Path(TMP_DIR)

if WIPE_TEMP_AFTER_RUN:
    # Only allow aggressive wiping if we are *not* using the system temp dir and we explicitly redirected temp:
    if TEMP_REDIRECT_VALID:
        if not tmp_root.exists():
            print(f"[NUKE] Nothing to do: {tmp_root} does not exist.")
        else:
            try:
                shutil.rmtree(tmp_root)
                print(f"[NUKE] Removed {tmp_root} and all contents.")
            except Exception as e:
                print(f"[WARN] Could not remove {tmp_root}: {e}")
        # Recreate empty tmp for future runs:
        try:
            tmp_root.mkdir(parents=True, exist_ok=True)
            print(f"[NUKE] Recreated empty tmp at: {tmp_root}")
        except Exception as e:
            print(f"[WARN] Could not recreate {tmp_root}: {e}")
    else:
        print(
            "[SAFEGUARD] WIPE_TEMP_AFTER_RUN=True but TEMP_REDIRECT_VALID=False. "
            "Refusing to wipe TMP_DIR to avoid deleting shared/system temp.")
else:
    print(f"[TMP] WIPE_TEMP_AFTER_RUN=False -> leaving TMP_DIR untouched at: {tmp_root}")

--------